# 信令数据验收

## 开工前：第一次封装

正常情况下：你需要把D01里面的代码配置和dq_record整段复制一遍，但是这样子明显不方便,所以你需要更好的处理方法, 该把他收起来

什么时候要进行封装：
- T1: 同一个代码出现3次，就得考虑封装了
- T2: 另外一个notebook使用它
- T3：口径或规范：第二次出现就收， 原因:口径得写两份，两份不一致代价，比代码重复大得多
- T4：新的需求：要重复运行，给别人运行，要测试

### 路径与口径常量

可以将路径或者口径常量放入的config.py里面

src/signal_insight/config.py
```python
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[2]

DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "signal.db"  # 探索库
# 检查库，前一天没做完可以从这里恢复
CHECKPOINT_DB = DATA_DIR / "checkpoints.db"

IMAGE_DIR = PROJECT_ROOT / "image"

# 口径常量：
UTC_OFFSET_HOURS = 8
ANALYSIS_DAY = "2018-10-03"
SOURCE_NOTE = "数据来源：沈阳移动信令样本(脱敏，2018年国庆)"
AUTHOR = "AzureKite 1210"
```

In [ ]:
# %run ../src/signal_insight/config.py  不推荐
from signal_insight import config

In [ ]:
print(config.AUTHOR)

需求：目前config.py存在一个问题，如果他的位置一旦发生变化，那么PROJECT这个常量的值就会发生变化，从而导致路径出现问题，因此需要一种机制，不管config.py放在哪里，都能找到正确的项目根路径

### 打包的处理

核心目的：为了在其他位置可以更好的使用可以定义的环境变量函数这些、。不做的的话，每天都要把前一天的代码复制一次，如果封装为函数或者常量做成 文件放在那里不打包，新的notebook文件，凡事涉及到修改封装好的函数，每次使用都要重启内核，就很麻烦，所以我们做成包，方便导入

1. 不管任何情况都要使用虚拟环境： signal 这个虚拟环境，因此打开终端第一步：激活虚拟环境
2. 你终端里面的工作路径，必须是你项目的根路径
3. pycharm打开项目，注意打开的目录必须是项目的根目录. 此外pycharm当前项目的解释器必须是我们创建的解释器

4. 创建src目录，创建signal_insight 软件包,后者比前者多了一个`__init__.py`
5. 在项目`根目录`创建pyproject.toml文件，并且按照要求写入内容
    ```toml
    [project]
    name = "signal-insight"   # 发行名：pip 用，习惯用短横线
    version = "0.1.0"
    description = "用于快速构建和分析信令数据" # 包的描述
    readme = "README.md" # 项目的说明文件
    requires-python = ">=3.12"
    license = "MIT"
    authors = [{ name = "Azurekite", email = "sx12101184@qq.com" }]
    keywords = ["信令数据分析", "pandas", "matplotlib", "teaching"]
    classifiers = [
        "Programming Language :: Python :: 3",
        "Operating System :: OS Independent",
        "Intended Audience :: Education",
        "Environment :: Console",
    ]

    dependencies = [
        "pandas>=3.0.6",
        "numpy>=2.5.3",
        "matplotlib>=3.11.2"
    ]

    [build-system]
    requires = ["hatchling"]
    build-backend = "hatchling.build"

    [tool.hatch.build.targets.wheel]
    packages = ["src/signal_insight"]
    ```
6. 打开终端：在激活虚拟环境和当前的位置位于项目根目录的前提下：执行安装`pip install -e .`
7. 在src目录下创建config.py然后写入内容，主要是将常用的变量写入进去
```python
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# @File       : config.py
# @Path       : signal-insight/src/signal_insight/config.py
# @Project    : signal-insight
# @Author     : Matri
# @Date       : 2026-10-10 13:12:07
# @Version    : v1.0.0
# @Description: 
# @ChangeLog:
#   2026-10-10 13:12:07 | Matri | v1.0.0 | 初始化创建
from pathlib import Path

# 项目根的标记文件：根目录里只要有这个文件，就能锁定项目根，
# 不再依赖 config.py 本身所在的层级（parents 数量）。
WORKSPACE_MARKER = "signal.workspace"

def find_project_root(marker: str = WORKSPACE_MARKER) -> Path:
    """从 config.py 所在目录逐级向上查找含 marker 的目录，作为项目根路径。

    这样即使 config.py 以后移动到包内更深的层级，
    也能靠根目录里的 signal.workspace 标记文件找到正确的项目根。
    """
    for parent in Path(__file__).resolve().parents:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"未找到项目根标记文件 {marker!r}，请确认它存在于项目根目录"
    )

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "signal.db"  # 探索库
# 检查库，前一天没做完可以从这里恢复
CHECKPOINT_DB = DATA_DIR / "checkpoints.db"

IMAGE_DIR = PROJECT_ROOT / "image"

# 口径常量：
UTC_OFFSET_HOURS = 8
ANALYSIS_DAY = "2018-10-03"
SOURCE_NOTE = "数据来源：沈阳移动信令样本(脱敏，2018年国庆)"
AUTHOR = "AzureKite 1210"
```

### 字体，配色出图收进config.py

In [ ]:
from signal_insight import viz
font = viz.apply_theme(theme="lively")

In [ ]:
import matplotlib.pyplot as plt
# X轴：月份 (1-12月)
x = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
# 华东区 (y1)
y1 = [62, 68, 76, 72, 80, 88, 93, 90, 98, 105, 110, 121]
# 华南区 (y2)
y2 = [48, 51, 50, 58, 60, 59, 66, 72, 69, 76, 82, 85]

fig, ax = plt.subplots(figsize=(10, 5),dpi=100, layout='constrained')
l1, = ax.plot(x, y1, 
#         color='#1F4E79',
        linewidth = 1.2,
        linestyle = '-',
        marker = 'o',
        markerfacecolor='white',
#         markeredgecolor='black',
        markersize = 4,
        alpha=1,
        label='华东区')
l2, = ax.plot(x, y2, 
            ls = '--' ,  # 等价于linestyle
#             c = 'C0', # 颜色等价于color
            lw = 1.6,  # 等价于linewidth
            marker = 's', 
            ms = 4, # markersize
            mfc = 'white', #markerfacecolor
#             mec = 'red',  # markeredegecolor 
            label='华南区')
ax.set_xticks(x, [f'{m}月' for m in x])
ax.set_ylabel('销售额（万元）')
ax.set_xlabel('月份')
ax.legend(loc=0) # 设定图例的位置

viz.finish(fig=fig,
    title='华南区华北区GMV情况',
    subtitle='',
    source=f"这是测试数据,数据来源xxx, 制图: {viz.config.AUTHOR}",
    path='这是测试图片'
          )
plt.show()

In [ ]:
# 目前是一套固定的主题,配色比较单调,虽然节省了我们很多的配置时间,但是视觉效果有待加强

### 稽核函数修改